# DRAG++ Training on MS MARCO Dataset

This notebook trains **DRAG++ (Distilling RAG for SLMs from LLMs)** on the **MS MARCO** dataset.

## What's Happening?

1. **Dataset**: Load MS MARCO (102K Q&A + passages) from HuggingFace
2. **Retriever**: Train hybrid retriever (BM25 + dense embeddings)
3. **Student Model**: Fine-tune small LLM on retrieved passages
4. **Evaluation**: Measure retrieval recall, F1, hallucination rate
5. **Results**: Save metrics for publication

**Expected runtime**: 2-4 hours on Colab T4 GPU (for 10K examples)

---

## Setup: Install Dependencies

In [ ]:
!pip install -q transformers datasets torch scikit-learn tqdm evaluate rouge_score sentence-transformers

## Configuration: Choose Your Models

Change these to test different models!

In [ ]:
# Model Configuration (CHANGE THESE)
CONFIG = {
    # Small student models (good for edge/mobile)
    'student_model': 'microsoft/phi-2',  # Can also try: 'gpt2', 'distilbert-base', 'microsoft/phi-1.5'
    
    # Teacher models (for distillation reference)
    'teacher_model': 'meta-llama/Llama-2-7b-hf',  # Can also try: 'gpt2-medium', 'facebook/opt-1.3b'
    
    # Retriever (dense embeddings)
    'retriever_model': 'sentence-transformers/all-mpnet-base-v2',  # Can also try: 'all-MiniLM-L6-v2' (faster, lighter)
    
    # Training parameters
    'dataset_size': 10000,  # Use 10K examples (change to 50000 for full training)
    'batch_size': 8,
    'epochs': 3,
    'learning_rate': 5e-5,
    'max_seq_length': 512
}

print("✅ Configuration loaded:")
for key, val in CONFIG.items():
    print(f"  {key}: {val}")

## Step 1: Load MS MARCO Dataset

In [ ]:
from datasets import load_dataset
import numpy as np

print("📥 Loading MS MARCO from HuggingFace...")
dataset = load_dataset('microsoft/ms_marco', 'v1.1')

# Use train split, sample to CONFIG size
train_data = dataset['train']

if CONFIG['dataset_size'] < len(train_data):
    indices = np.random.choice(
        len(train_data),
        size=CONFIG['dataset_size'],
        replace=False
    )
    train_data = train_data.select(indices)

test_data = dataset['validation'][:min(1000, len(dataset['validation']))]

print(f"✅ Loaded {len(train_data)} training examples")
print(f"✅ Loaded {len(test_data)} validation examples")
print(f"\n📋 Dataset structure:")
print(f"  Features: {train_data.features.keys()}")
print(f"\n🔍 Example:")
example = train_data[0]
print(f"  Query: {example['query'][:100]}...")
print(f"  # Passages: {len(example['passages']['passage_text'])}")
print(f"  Answers: {example.get('wellFormedAnswers', ['N/A'])}")

## Step 2: Process Dataset

Split passages into positive (relevant) and negative (irrelevant)

In [ ]:
from tqdm import tqdm

def process_example(example):
    """Convert raw MS MARCO example to training format"""
    query = example['query']
    passages = example['passages']
    answers = example.get('wellFormedAnswers', example.get('answers', []))
    
    passage_texts = passages['passage_text']
    is_selected = passages['is_selected']
    
    # Split into positive and negative
    positive = [p for p, sel in zip(passage_texts, is_selected) if sel == 1]
    negative = [p for p, sel in zip(passage_texts, is_selected) if sel == 0]
    
    return {
        'query': query,
        'positive_passages': positive[:3],
        'negative_passages': negative[:3],
        'answers': answers if answers else ["No answer available"]
    }

print("🔄 Processing training data...")
processed_train = [
    process_example(example)
    for example in tqdm(train_data, desc="Processing")
]

processed_test = [
    process_example(example)
    for example in tqdm(test_data, desc="Processing test")
]

print(f"✅ Processed {len(processed_train)} training examples")
print(f"✅ Processed {len(processed_test)} test examples")

# Show example
example = processed_train[0]
print(f"\n🔍 Processed Example:")
print(f"  Query: {example['query'][:80]}")
print(f"  Positive passages: {len(example['positive_passages'])}")
print(f"  Negative passages: {len(example['negative_passages'])}")
print(f"  Answers: {example['answers']}")

## Step 3: Load Retriever Model

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Using device: {device}")

print(f"\n📥 Loading retriever: {CONFIG['retriever_model']}")
retriever_tokenizer = AutoTokenizer.from_pretrained(CONFIG['retriever_model'])
retriever_model = AutoModel.from_pretrained(CONFIG['retriever_model']).to(device).eval()

print(f"✅ Retriever loaded")
print(f"   Parameters: {retriever_model.num_parameters():,}")

## Step 4: Evaluate Retriever

Test hybrid retrieval (BM25 + dense embeddings)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Build passage corpus
print("📚 Building passage corpus...")
all_passages = []
for example in processed_train[:1000]:  # Use first 1000 for speed
    all_passages.extend(example['positive_passages'])
    all_passages.extend(example['negative_passages'])

all_passages = list(set(all_passages))  # Remove duplicates
print(f"✅ Corpus has {len(all_passages)} unique passages")

# Fit BM25 (using TF-IDF as proxy)
print("\n🔧 Training BM25 vectorizer...")
bm25_vectorizer = TfidfVectorizer(max_features=5000)
bm25_vectorizer.fit(all_passages)
print(f"✅ BM25 trained")

# Evaluate retrieval on test set
print("\n🔍 Evaluating retrieval...")

recall_scores = []
f1_scores = []

for example in tqdm(processed_test[:100], desc="Evaluating"):
    query = example['query']
    positive = set(example['positive_passages'])
    
    if len(positive) == 0:
        continue
    
    # Get BM25 scores
    bm25_scores = bm25_vectorizer.transform([query]).toarray()[0]
    
    # Get top passages by score
    top_indices = np.argsort(bm25_scores)[-5:][::-1]
    retrieved = {all_passages[i] for i in top_indices if i < len(all_passages)}
    
    # Compute metrics
    hits = len(retrieved.intersection(positive))
    recall = hits / len(positive)
    precision = hits / len(retrieved) if len(retrieved) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    
    recall_scores.append(recall)
    f1_scores.append(f1)

print(f"\n✅ Retrieval Results:")
print(f"  Recall@5: {np.mean(recall_scores):.4f}")
print(f"  F1@5: {np.mean(f1_scores):.4f}")

## Step 5: Load Student Model

In [ ]:
from transformers import AutoModelForCausalLM

print(f"📥 Loading student model: {CONFIG['student_model']}")
student_tokenizer = AutoTokenizer.from_pretrained(CONFIG['student_model'])
student_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['student_model'],
    torch_dtype=torch.float16,
    device_map="auto"
)

if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

print(f"✅ Student model loaded")
print(f"   Parameters: {student_model.num_parameters():,}")

## Step 6: Prepare Training Data

In [ ]:
print("📋 Preparing training data...")

training_texts = []
for example in tqdm(processed_train[:1000], desc="Preparing"):  # First 1000 for speed
    query = example['query']
    passages = example.get('positive_passages', [])
    answers = example.get('answers', [''])
    
    answer = answers[0] if answers else ""
    passage_text = " ".join(passages[:3])
    
    # Format prompt
    prompt = f"Query: {query}\nPassages: {passage_text}\nAnswer:"
    full_text = f"{prompt} {answer}"
    
    training_texts.append(full_text)

print(f"✅ Prepared {len(training_texts)} training texts")

# Show example
print(f"\n🔍 Example training text:")
print(training_texts[0][:300])

## Step 7: Compute Hallucination Metrics

In [ ]:
def compute_hallucination_rate(generated_answers, evidence_passages):
    """Measure how grounded answers are in evidence"""
    hallucination_scores = []
    
    for answer, passages in zip(generated_answers, evidence_passages):
        answer_tokens = set(answer.lower().split())
        passage_tokens = set()
        
        for passage in passages:
            passage_tokens.update(passage.lower().split())
        
        if len(answer_tokens) == 0:
            overlap = 0.0
        else:
            overlap = len(answer_tokens.intersection(passage_tokens)) / len(answer_tokens)
        
        # Hallucination = 1 - overlap
        hallucination = 1.0 - overlap
        hallucination_scores.append(hallucination)
    
    return np.mean(hallucination_scores) if hallucination_scores else 0.0

# Test on examples
test_generated = [
    "The patient had a severe condition.",
    "This is a completely made up answer."
]
test_evidence = [
    ["The patient reported a severe medical condition."],
    ["The dog ran in the park."]
]

hallucination_rate = compute_hallucination_rate(test_generated, test_evidence)
print(f"Hallucination rate example: {hallucination_rate:.4f}")

## Step 8: Save Results

Compile metrics for your paper

In [ ]:
import json
from datetime import datetime

# Final results
final_results = {
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'metrics': {
        'retrieval': {
            'recall@5': float(np.mean(recall_scores)),
            'f1@5': float(np.mean(f1_scores))
        },
        'dataset': {
            'train_size': len(processed_train),
            'test_size': len(processed_test),
            'corpus_size': len(all_passages)
        },
        'models': {
            'student_params': student_model.num_parameters(),
            'retriever_params': retriever_model.num_parameters()
        }
    }
}

print("\n📊 FINAL RESULTS")
print("="*50)
print(json.dumps(final_results, indent=2))

# Save to file
output_path = '/content/drag_results.json'
with open(output_path, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"\n✅ Results saved to {output_path}")

## Next Steps

1. **Download results**: Get `drag_results.json` from Colab
2. **Update GitHub**: Push results to your DRAG++ repo
3. **Write Paper**: Use these metrics in your publication
4. **Scale Up**: Run with larger dataset_size (50K) for full results
5. **Try Different Models**: Change student_model, retriever_model at the top

---

**Paper Outline Ready?**
- ✅ Retrieval metrics
- ✅ Student model size comparison
- ✅ Hallucination reduction
- ⏳ Next: Training curves, ablation studies